# Class 2 — Embeddings & Vector Databases

**Week 6: Foundations of RAG and Chatbots**

### Learning objectives
By the end of this notebook you will be able to:
- Explain what an embedding is and why similar meanings land near each other in vector space
- Compute cosine similarity between two vectors by hand and with NumPy
- Generate real embeddings via an API and rank a small set of documents against a query
- Describe, at a high level, what a vector database (Chroma, FAISS, Pinecone) adds beyond a plain list

> This notebook contains a fully runnable demo: real embeddings, real cosine similarity, a real nearest-neighbor
> match. All you need is one API key.

## Setup

As in Class 1, we read the API key from the environment — never hardcode a key in a notebook.

In [ ]:
import os

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY")

if not OPENAI_API_KEY:
    print(
        "No OPENAI_API_KEY found in your environment.\n"
        "This notebook's embeddings demo uses OpenAI's embeddings endpoint, since it is the most widely "
        "available embeddings API at the time of writing.\n"
        "Set it with:  export OPENAI_API_KEY=sk-...\n"
        "The rest of the notebook will still explain the concepts, but the embedding cells will raise an "
        "error until a key is set."
    )
else:
    print("Found OPENAI_API_KEY. You're ready to run the embeddings demo below.")

if ANTHROPIC_API_KEY:
    print("Found ANTHROPIC_API_KEY too — you can reuse the call_llm-style pattern from Class 1 for the generation half in Class 3.")

## Concept 1 — What Is an Embedding, Really?

An embedding is a list of numbers (a vector) that represents the *meaning* of a piece of text. A trained embedding
model turns text into this vector. Two texts with similar meaning end up with vectors that are close together in
that numeric space — "dog" and "puppy" land near each other, while "dog" and "stock market" land far apart.

Real embedding models produce vectors with hundreds or thousands of numbers (dimensions). We can't draw that, but
the geometry idea is identical to the small examples below.

## Concept 2 — Measuring Similarity With Cosine

Before we call any API, let's build intuition with tiny, fake 2-dimensional "embeddings" we can plot mentally.
Cosine similarity measures the angle between two vectors — it asks "do they point in the same direction?" — and
ignores their length. Scores run roughly from -1 (opposite) to 1 (identical direction).

In [ ]:
import numpy as np

def cosine_similarity(a, b):
    a = np.array(a, dtype=float)
    b = np.array(b, dtype=float)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

# Tiny made-up 2D "embeddings" just to build intuition -- real embeddings have hundreds of dimensions.
dog    = [0.9, 0.2]
puppy  = [0.85, 0.3]
stocks = [-0.1, 0.95]

print("dog vs puppy: ", cosine_similarity(dog, puppy))
print("dog vs stocks:", cosine_similarity(dog, stocks))

Notice "dog" and "puppy" score much higher (closer to 1) than "dog" and "stocks" — exactly what we'd hope for.
Now let's do this with **real** embeddings from an API instead of numbers we made up.

## Concept 3 — Generate Real Embeddings and Rank Documents

We'll embed a small set of documents and one query using OpenAI's embeddings endpoint, then rank the documents
by cosine similarity to the query. The highest-scoring document is the nearest neighbor — the one we'd retrieve
in a real RAG system.

In [ ]:
from openai import OpenAI

EMBEDDING_MODEL = "text-embedding-3-small"

def get_embedding(text, model=EMBEDDING_MODEL):
    """Return the embedding vector (a list of floats) for a piece of text."""
    client = OpenAI(api_key=OPENAI_API_KEY)
    response = client.embeddings.create(model=model, input=text)
    return response.data[0].embedding

documents = [
    "Password reset instructions for the employee portal: click 'Forgot password' and check your email.",
    "Our refund policy allows returns within 30 days of purchase with a valid receipt.",
    "Office hours are 9am to 5pm, Monday through Friday, excluding public holidays.",
]

query = "How do I reset my password?"

# Embed the query and every document
query_vec = get_embedding(query)
doc_vecs = [get_embedding(doc) for doc in documents]

# Rank documents by cosine similarity to the query
scored = sorted(
    zip(documents, doc_vecs),
    key=lambda pair: cosine_similarity(query_vec, pair[1]),
    reverse=True,
)

for rank, (doc, vec) in enumerate(scored, start=1):
    score = cosine_similarity(query_vec, vec)
    print(f"{rank}. [{score:.3f}] {doc}")

The top-ranked document should be the password-reset one, even though the query never uses the exact word
"reset" the way the document text might. The embedding model matched **meaning**, not literal words — this is the
core trick behind semantic search.

## Concept 4 — Semantic Search vs. Keyword Search

- **Keyword search** matches exact words. A search for "puppy food" can miss an article titled "canine nutrition"
  even though it's exactly what the user wants.
- **Semantic search** (what we just did above) matches meaning. It handles synonyms, paraphrasing, and typos far
  more gracefully.
- Many production systems combine both — "hybrid search" — to get literal-match precision plus semantic recall.

## Concept 5 — Vector Databases: Where Embeddings Live

For three documents, a Python list is plenty. For a million documents, you need a system built to find nearest
neighbors fast. A few common options:

- **Chroma** — open-source, runs locally or embedded in your app; great for prototyping.
- **FAISS** — Meta's similarity-search library; in-memory and extremely fast; no server, you manage persistence.
- **Pinecone** — fully-managed cloud vector database; scales to billions of vectors; ops handled for you.

All three do the same core job as the `sorted(...)` call above: store vectors, and find the nearest ones to a
query vector, just at a much larger scale and with far more engineering underneath.

## Challenges

Each of these builds directly on the `get_embedding` and `cosine_similarity` functions above.

### Challenge 1 — Find the Odd One Out

Write five short sentences of your own, where four share a topic and one is unrelated. Embed all five, compute the
average pairwise similarity of each sentence to the other four, and print which one is the "odd one out."

**Acceptance criteria:** your code correctly identifies the unrelated sentence using similarity scores, not by
just eyeballing it.

In [ ]:
# TODO: write 5 sentences (4 related, 1 unrelated), embed them, and find the odd one out


### Challenge 2 — Swap the Query

Using the same `documents` list from Concept 3, write a new query that should match the refund-policy document
instead, and print the ranked results.

**Acceptance criteria:** the refund-policy document ranks first for your new query.

In [ ]:
# TODO: write a new query and print the re-ranked documents


### Challenge 3 — Break It

Craft a query that you predict will fool cosine similarity into ranking an irrelevant document highly (for example,
a query that shares vocabulary with the wrong document but means something different). Run it and report whether
your prediction was correct.

**Acceptance criteria:** you state your prediction *before* running the code, then compare it to the actual
ranking.

In [ ]:
# TODO: craft a tricky query, predict the outcome in a comment, then run and compare


### Challenge 4 — Cosine vs. Euclidean

Write a `euclidean_distance(a, b)` function (straight-line distance between two vectors) and compare its ranking
of the `documents` list against `cosine_similarity`'s ranking for the same query. Are the rankings the same?

**Acceptance criteria:** you print both rankings side by side and state in a comment whether they agree.

In [ ]:
# TODO: implement euclidean_distance and compare its ranking to cosine_similarity's ranking


### Challenge 5 — Bonus: Make the Case for a Vector Database

In a comment or markdown-style triple-quoted string, argue in three sentences which of Chroma, FAISS, or Pinecone
you'd pick for a small internal support-chatbot project with under 5,000 documents, and why.

**Acceptance criteria:** your answer names one specific tradeoff (setup cost, scale, or ops burden) that drove
your choice.

In [ ]:
# TODO: argue for one vector database choice in 3 sentences, referencing a specific tradeoff
